In [14]:
import pandas as pd
import glob
import os


df_clusters = pd.read_csv(r"C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\datos\LLM\municipios_etiquetados_niveles.csv")

df_clusters["municipio"] = df_clusters["municipio"].str.strip().str.lower()
df_clusters["isla"] = df_clusters["isla"].str.strip().str.lower()

In [15]:

def cargar_datasets_ocio(base_path):
    data = []

    for isla_path in glob.glob(os.path.join(base_path, "*")):
        isla = os.path.basename(isla_path).replace("_ocio", "").lower()

        clean_path = os.path.join(isla_path, "clean_dataset")

        if not os.path.exists(clean_path):
            continue

        for file in glob.glob(os.path.join(clean_path, "*.csv")):
            df = pd.read_csv(file)

            df["municipio"] = df.iloc[:, 0].str.strip().str.lower()
            df["isla"] = isla

            tipo = None
            filename = os.path.basename(file)

            if "alojamientos" in filename:
                tipo = "alojamientos"
                df.rename(columns={"nombre": "alojamiento", "tipo": "alojamiento_tipo", "url_maps": "alojamiento_url"}, inplace=True)

            elif "restaurantes" in filename:
                tipo = "restaurantes"
                df.rename(columns={"nombre": "restaurante", "grupo": "restaurante_tipo", "url_maps": "restaurante_url"}, inplace=True)

            elif "playas" in filename:
                tipo = "playas"
                df.rename(columns={"playa": "playa", "url_maps": "playa_url"}, inplace=True)

            df["tipo_dataset"] = tipo
            data.append(df)

    return pd.concat(data, ignore_index=True)

In [16]:
import unicodedata

def normalizar_municipio(nombre):
    if pd.isna(nombre):
        return nombre

    nombre = nombre.lower().strip()

    nombre = unicodedata.normalize("NFKD", nombre)
    nombre = nombre.encode("ascii", "ignore").decode("utf-8")

    nombre = " ".join([w.capitalize() for w in nombre.split()])

    return nombre

In [17]:
map_islas = {
    "el_hierro": "El Hierro",
    "fuerteventura": "Fuerteventura",
    "grancanaria": "Gran Canaria",
    "gran canaria": "Gran Canaria",
    "lanzarote": "Lanzarote",
    "la_gomera": "La Gomera",
    "la gomera": "La Gomera",
    "la_palma": "La Palma",
    "la palma": "La Palma",
    "tenerife": "Tenerife"
}

def normalizar_isla(x):
    x = str(x).strip().lower()
    return map_islas.get(x, x.title())

In [18]:
def corregir_municipios_especiales(nombre):
    if not isinstance(nombre, str):
        return nombre

    nombre_l = nombre.lower().strip()

    if nombre_l in ["valsequillo", "valsequillo de gran canaria"]:
        return "Valsequillo"

    if nombre_l in [
        "santa maria de guia",
        "santa maria de guia de gran canaria"
    ]:
        return "Santa Maria De Guia"

    if nombre_l in [
        "vega de san mateo",
        "vega san mateo"
    ]:
        return "Vega De San Mateo"

    return nombre

In [19]:
df_ocio = cargar_datasets_ocio(r"C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\datos\LLM")

df_clusters["municipio"] = df_clusters["municipio"].apply(normalizar_municipio)
df_clusters["isla"] = df_clusters["isla"].apply(normalizar_municipio)

df_ocio["municipio"] = df_ocio["municipio"].apply(normalizar_municipio)
df_ocio["isla"] = df_ocio["isla"].apply(normalizar_municipio)

df_clusters["municipio"] = df_clusters["municipio"].apply(corregir_municipios_especiales)
df_ocio["municipio"] = df_ocio["municipio"].apply(corregir_municipios_especiales)

df_clusters["isla"] = df_clusters["isla"].apply(normalizar_isla)
df_ocio["isla"] = df_ocio["isla"].apply(normalizar_isla)


df_aloj = df_ocio[df_ocio["tipo_dataset"] == "alojamientos"]
df_rest = df_ocio[df_ocio["tipo_dataset"] == "restaurantes"]
df_playa = df_ocio[df_ocio["tipo_dataset"] == "playas"]

In [20]:
df_aloj["isla"].unique()

<StringArray>
[    'El Hierro', 'Fuerteventura',  'Gran Canaria',     'Lanzarote',
     'La Gomera',      'La Palma',      'Tenerife']
Length: 7, dtype: str

In [21]:
def agrupar(df, columnas):
    return df.groupby(["municipio", "isla"]).apply(
        lambda x: x[columnas].to_dict(orient="records")
    ).reset_index(name=columnas[0] + "_list")

aloj_group = agrupar(df_aloj, ["alojamiento", "alojamiento_tipo", "alojamiento_url"])
rest_group = agrupar(df_rest, ["restaurante", "restaurante_tipo", "restaurante_url"])
playa_group = agrupar(df_playa, ["playa", "playa_url"])

In [22]:
df_final = (
    df_clusters
    .merge(aloj_group, on=["municipio", "isla"], how="left")
    .merge(rest_group, on=["municipio", "isla"], how="left")
    .merge(playa_group, on=["municipio", "isla"], how="left")
)

df_final

,municipio,isla,nivel_turistico_municipio,nivel_turistico_medio,n_clusters,clusters_ids,n_fotos_turistico,n_fotos_local,alojamiento_list,restaurante_list,playa_list
0,Adeje,Tenerife,2,2.45,33,"51, 65, 68, 93, 125, 138, 139, 141, 142, 143, ...",1044,1074,"[{'alojamiento': 'PUEBLO TORVISCAS', 'alojamie...","[{'restaurante': 'CHEZ MIMI', 'restaurante_tip...","[{'playa': 'Las Vistas', 'playa_url': 'https:/..."
1,Agaete,Gran Canaria,3,2.82,11,"61, 62, 64, 70, 72, 84, 85, 86, 102, 103, 104",336,179,[{'alojamiento': 'HOTEL OCCIDENTAL ROCA NEGRA'...,"[{'restaurante': 'Casa Lolo', 'restaurante_tip...","[{'playa': 'Las Nieves', 'playa_url': 'https:/..."
2,Agulo,La Gomera,3,3.14,7,"10, 13, 14, 16, 17, 21, 22",290,117,"[{'alojamiento': 'APARTAMENTOS BAJIP', 'alojam...","[{'restaurante': 'Avenida', 'restaurante_tipo'...","[{'playa': 'Playa de Agulo', 'playa_url': 'htt..."
3,Aguimes,Gran Canaria,3,2.90,10,"0, 1, 2, 13, 15, 16, 18, 19, 23, 29",430,160,"[{'alojamiento': 'HOTEL PLAYA DE ARINAGA', 'al...","[{'restaurante': 'Del Pino', 'restaurante_tipo...","[{'playa': 'Playa de Arinaga', 'playa_url': 'h..."
4,Alajero,La Gomera,2,2.33,3,"0, 1, 7",143,167,"[{'alojamiento': 'APARTAMENTOS CASANOVA I', 'a...","[{'restaurante': 'Bar Restaurante La Chalana',...","[{'playa': 'Playa de Santiago', 'playa_url': '..."
...,...,...,...,...,...,...,...,...,...,...,...
73,Valverde,El Hierro,4,3.80,10,"0, 3, 4, 5, 6, 13, 16, 17, 18, 19",469,28,"[{'alojamiento': 'HOTEL BOOMERANG', 'alojamien...","[{'restaurante': 'Disco-Pub ""La Lonja""', 'rest...","[{'playa': 'Playa de Timijiraque', 'playa_url'..."
74,Vega De San Mateo,Gran Canaria,3,2.60,5,"73, 74, 75, 87, 97",260,141,[{'alojamiento': 'Casa Rural El Pajar de Migue...,"[{'restaurante': 'Mallow', 'restaurante_tipo':...",NaN
75,Vilaflor,Tenerife,4,4.00,4,"70, 76, 83, 84",252,14,"[{'alojamiento': 'VILLALBA', 'alojamiento_tipo...","[{'restaurante': 'MIRADOR, EL', 'restaurante_t...",NaN
76,Villa De Mazo,La Palma,4,3.67,6,"17, 35, 36, 37, 40, 49",373,57,"[{'alojamiento': 'HOTEL ARMINDA', 'alojamiento...","[{'restaurante': 'Central', 'restaurante_tipo'...","[{'playa': 'Playa de La Salemera', 'playa_url'..."


In [23]:
df_final["isla"].unique()

<StringArray>
[     'Tenerife',  'Gran Canaria',     'La Gomera', 'Fuerteventura',
     'Lanzarote',      'La Palma',     'El Hierro']
Length: 7, dtype: str

In [24]:
df_final.to_csv(r"C:\Users\cgsos\Documents\Cuarto\TFG\FLICKR\datos\LLM\Dataset\dataset_final_islas_canarias.csv", index=False)